In [1]:
import random
from pathlib import Path
from itertools import product

import numpy as np
import pandas as pd

import pyarrow as pa
import pyarrow.parquet as pq

import matplotlib.pyplot as plt
import seaborn as sns

from scipy.stats import norm

from sklearn.linear_model import Lasso
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

import statsmodels.api as sm
import statsmodels.formula.api as smf

import optuna

import seaborn as sns

from stage1 import lasso_rolling_window, calculate_r_squared
from stage2 import estimate_kappa_curve_fit, compute_alm_returns, compute_stage2_r_squared
from grid_search import grid_search, estimate_single_config

import plot_style
plot_style.apply()


In [2]:
# load feature matrix and response variable
feature_matrix = pd.read_csv("../../data/merged_return_topic_data.csv", index_col=0, parse_dates=True)
response_variables = pd.read_csv("../../data/response.csv", index_col=0, parse_dates=True)
stock_data = pd.read_csv("../../data/X.csv", index_col=0, parse_dates=True)

FileNotFoundError: [Errno 2] No such file or directory: '../../data/merged_return_topic_data.csv'

Summary Statistics of Data


In [3]:
## seperate topic from stock data ##

# boolean mask: columns that contain any letter
has_letters = stock_data.columns.str.contains('[a-zA-Z]')

# topic-related columns (strings, topic names, etc.)
topic_data = stock_data.loc[:, has_letters].copy()

# stock-related columns (numeric identifiers, PERMNOs, etc.)
stock_data_only = stock_data.loc[:, ~has_letters].copy()

In [4]:
## Functions to create Summary Data Table ##

def to_ar1_innovations(X: pd.DataFrame, min_obs: int = 30, 
                       print_summary: bool = True, return_coefs: bool = False):
    """Return AR(1) residuals for each column of X."""
    innovations = pd.DataFrame(index=X.index, columns=X.columns, dtype=float)
    coefs = {}

    for col in X.columns:
        s = pd.to_numeric(X[col], errors='coerce')
        df = pd.DataFrame({'x': s, 'lag': s.shift(1)}).dropna()
        
        if len(df) < min_obs or df['x'].nunique() < 3:
            continue
            
        model = sm.OLS(df['x'], sm.add_constant(df['lag'])).fit()
        innovations.loc[df.index, col] = model.resid
        coefs[col] = model.params['lag']

    coefs = pd.Series(coefs)

    if print_summary and not coefs.empty:
        stats = pd.Series({
            'N': len(coefs), 'Mean': coefs.mean(), 'Std': coefs.std(),
            'Min': coefs.min(), 'P5': coefs.quantile(0.05),
            'Median': coefs.median(), 'P95': coefs.quantile(0.95),
            'Max': coefs.max()
        })
        print(stats.to_frame().T.to_latex(float_format='%.4f'))

    return (innovations, coefs) if return_coefs else innovations


def panel_construction(stock_data: pd.DataFrame) -> pd.DataFrame:
    """Panel A: log stock return statistics."""
    r = np.log1p(stock_data).stack()
    
    stats = pd.DataFrame({
        'Mean': [r.mean()], 'Std': [r.std()], 'Min': [r.min()], 
        'Max': [r.max()], 'Skew': [r.skew()], 'Kurt': [r.kurtosis()], 
        'Obs': [r.count()]
    })
    
    print(stats.to_latex(float_format='%.4f'))
    return stats


def panel_b_ar1_phi(ar1_phi: pd.Series) -> pd.DataFrame:
    """Panel B: AR(1) coefficient statistics."""
    s = pd.to_numeric(ar1_phi, errors='coerce').dropna()
    
    stats = pd.DataFrame({
        'Mean': [s.mean()], 'Std': [s.std()], 'Min': [s.min()],
        'Max': [s.max()], 'Skew': [s.skew()], 'Kurt': [s.kurtosis()],
        'Obs': [s.count()]
    }, index=[r'AR(1) coefficient $\phi$'])
    
    print(stats.to_latex(float_format='%.4f'))
    return stats


## Functions to get correlation matrix and table ##


def plot_corr_heatmap_png(df, output_file="correlation_heatmap", figsize=(10, 10), dpi=600):
    corr = df.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all").corr()
    
    sns.set_theme(style="white", font="serif")
    fig, ax = plt.subplots(figsize=figsize)
    sns.heatmap(corr, ax=ax, cmap="RdBu_r", vmin=-1, vmax=1, center=0, square=True, 
                cbar_kws={"label": "Correlation"})
    
    plt.savefig(f"{output_file}.png", dpi=dpi, bbox_inches="tight")
    plt.savefig(f"{output_file}.pdf", bbox_inches="tight")
    plt.close()


def cross_lag_correlation_summary(df, lags=(1, 2, 3)):
    X = df.apply(pd.to_numeric, errors="coerce").dropna(axis=1, how="all")
    results = []
    
    for k, l in product(lags, lags):
        Xk, Xl = X.shift(k), X.shift(l)
        valid = ~(Xk.isna().any(axis=1) | Xl.isna().any(axis=1))
        Xk_v, Xl_v = Xk[valid], Xl[valid]
        
        if len(Xk_v) < 2:
            results.append([k, l, np.nan, np.nan, np.nan, np.nan])
            continue
        
        Xk_std = (Xk_v - Xk_v.mean()) / Xk_v.std()
        Xl_std = (Xl_v - Xl_v.mean()) / Xl_v.std()
        corr = (Xk_std.values.T @ Xl_std.values) / len(Xk_std)
        corrs = corr[~np.eye(len(corr), dtype=bool)]
        corrs = corrs[~np.isnan(corrs)]
        
        if len(corrs) == 0:
            results.append([k, l, np.nan, np.nan, np.nan, np.nan])
        else:
            results.append([k, l, corrs.mean(), np.median(corrs), 
                          np.abs(corrs).max(), np.percentile(np.abs(corrs), 95)])
    
    return pd.DataFrame(results, columns=["k", "l", "Mean", "Median", "Max |r|", "P95 |r|"])

In [5]:
print("Topic data summary:")
panel_a = panel_construction(topic_data)
print("-----------------------------------------------------------------------")
print("Retun data summary:")
panel_a = panel_construction(stock_data_only)
print("-----------------------------------------------------------------------")
print("AR1 coefficients summary:")
X_innov, ar1_phi = to_ar1_innovations(topic_data, return_coefs=True)

Topic data summary:
\begin{tabular}{lrrrrrrr}
\toprule
 & Mean & Std & Min & Max & Skew & Kurt & Obs \\
\midrule
0 & 0.0055 & 0.0041 & 0.0000 & 0.1161 & 3.6616 & 33.3613 & 340020 \\
\bottomrule
\end{tabular}

-----------------------------------------------------------------------
Retun data summary:
\begin{tabular}{lrrrrrrr}
\toprule
 & Mean & Std & Min & Max & Skew & Kurt & Obs \\
\midrule
0 & 0.0003 & 0.0206 & -1.9641 & 2.0509 & -0.5867 & 101.9167 & 3060180 \\
\bottomrule
\end{tabular}

-----------------------------------------------------------------------
AR1 coefficients summary:
\begin{tabular}{lrrrrrrrr}
\toprule
 & N & Mean & Std & Min & P5 & Median & P95 & Max \\
\midrule
0 & 180.0000 & 0.1427 & 0.1404 & -0.0895 & -0.0020 & 0.0992 & 0.4290 & 0.7635 \\
\bottomrule
\end{tabular}



Corrleation Matrices and Coefficients

In [7]:
# plot_corr_heatmap_png(X_innov, output_file="corr_topics_heatmap.png")

In [8]:
summary_df = cross_lag_correlation_summary(X_innov, lags=(1, 3, 5, 7, 9, 11))
print(summary_df.to_latex(index=False, float_format="%.4f"))

\begin{tabular}{rrrrrr}
\toprule
k & l & Mean & Median & Max |r| & P95 |r| \\
\midrule
1 & 1 & -0.0003 & -0.0037 & 0.5453 & 0.0904 \\
1 & 3 & 0.0003 & -0.0001 & 0.2147 & 0.0567 \\
1 & 5 & 0.0002 & -0.0002 & 0.1653 & 0.0572 \\
1 & 7 & 0.0001 & -0.0004 & 0.1592 & 0.0567 \\
1 & 9 & 0.0002 & 0.0000 & 0.1514 & 0.0552 \\
1 & 11 & 0.0004 & 0.0001 & 0.2656 & 0.0598 \\
3 & 1 & 0.0003 & -0.0001 & 0.2147 & 0.0567 \\
3 & 3 & -0.0003 & -0.0037 & 0.5456 & 0.0904 \\
3 & 5 & 0.0003 & -0.0001 & 0.2146 & 0.0568 \\
3 & 7 & 0.0002 & -0.0001 & 0.1660 & 0.0573 \\
3 & 9 & 0.0001 & -0.0004 & 0.1594 & 0.0568 \\
3 & 11 & 0.0002 & -0.0001 & 0.1513 & 0.0552 \\
5 & 1 & 0.0002 & -0.0002 & 0.1653 & 0.0572 \\
5 & 3 & 0.0003 & -0.0001 & 0.2146 & 0.0568 \\
5 & 5 & -0.0003 & -0.0037 & 0.5455 & 0.0903 \\
5 & 7 & 0.0003 & -0.0000 & 0.2116 & 0.0568 \\
5 & 9 & 0.0002 & -0.0001 & 0.1658 & 0.0572 \\
5 & 11 & 0.0001 & -0.0004 & 0.1603 & 0.0566 \\
7 & 1 & 0.0001 & -0.0004 & 0.1592 & 0.0567 \\
7 & 3 & 0.0002 & -0.0001 & 0.1660 &

First Stage results

First stage estimation results table

In [10]:
import os
import pickle
import numpy as np
import pandas as pd

OUT_DIR = "results_full"  # change to your current run folder

def summarize(s):
    s = pd.to_numeric(s, errors="coerce").dropna()
    return pd.Series({
        "mean": s.mean(), "std": s.std(), "median": s.median(),
        "min": s.min(), "max": s.max(), "p05": s.quantile(0.05), "p95": s.quantile(0.95)
    })

# --- Load per-stock summaries (kept) ---
summary_dir = os.path.join(OUT_DIR, "summary")

summary_per_stock = {}
for fn in os.listdir(summary_dir):
    if not fn.endswith(".pkl"):
        continue
    stock = fn[:-4]
    with open(os.path.join(summary_dir, fn), "rb") as f:
        summary_per_stock[stock] = pickle.load(f)

# Keep only non-None summaries
summary_per_stock = {k: v for k, v in summary_per_stock.items() if v is not None}

# Build summaries dataframe (index = stock)
summaries = pd.DataFrame.from_dict(summary_per_stock, orient="index")

# --- R^2s ---
# Prefer the columns you used before if they exist; otherwise fall back to common alternatives.
def pick_col(df, candidates):
    for c in candidates:
        if c in df.columns:
            return c
    return None

col_r2_in = pick_col(summaries, ["r2_insample_stage1", "insample_r2", "r2_insample", "R2_in", "r2_in"])
col_r2_oos = pick_col(summaries, ["r2_oos_stage1", "oos_r2", "r2_oos", "R2_oos", "r2_out"])

r2_in = summarize(summaries[col_r2_in]) if col_r2_in else pd.Series(dtype=float)
r2_oos = summarize(summaries[col_r2_oos]) if col_r2_oos else pd.Series(dtype=float)

out = pd.DataFrame(
    [r2_in, r2_oos],
    index=["In-sample $R^2$", "Out-of-sample $R^2$"]
)

print(out.to_latex(float_format="%.4f"))

# --- Load compact objects (replaces details) ---
compact_dir = os.path.join(OUT_DIR, "compact")
compact_per_stock = {}

for fn in os.listdir(compact_dir):
    if not fn.endswith(".pkl"):
        continue
    stock = fn[:-4]
    with open(os.path.join(compact_dir, fn), "rb") as f:
        compact_per_stock[stock] = pickle.load(f)

# Keep only those also present in summaries (optional alignment)
compact_per_stock = {k: v for k, v in compact_per_stock.items() if k in summaries.index}

# Example: build a dataframe of stock-level compact scalars (variance + mean_num_active)
compact_stock_level = pd.DataFrame({
    "return_variance": {k: v.get("return_variance", np.nan) for k, v in compact_per_stock.items()},
    "mean_num_active": {k: v.get("mean_num_active", np.nan) for k, v in compact_per_stock.items()},
    "n_windows": {k: v.get("n_windows", np.nan) for k, v in compact_per_stock.items()},
})

# Example: if you want to summarise the time-level OOS R^2 series across stocks,
# take the last available value per stock:
last_oos_r2 = pd.Series({
    k: (np.nan if v.get("ts_oos_r2", np.array([])).size == 0 else float(pd.Series(v["ts_oos_r2"]).dropna().iloc[-1]))
    for k, v in compact_per_stock.items()
})
oos_r2_last_stats = summarize(last_oos_r2)

compact_stock_level, oos_r2_last_stats


\begin{tabular}{lrrrrrrr}
\toprule
 & mean & std & median & min & max & p05 & p95 \\
\midrule
In-sample $R^2$ & 0.1418 & 0.1596 & 0.0863 & -0.0040 & 0.7417 & -0.0020 & 0.4816 \\
Out-of-sample $R^2$ & -0.0351 & 0.0444 & -0.0177 & -0.3481 & 0.0032 & -0.1270 & -0.0008 \\
\bottomrule
\end{tabular}



(       return_variance  mean_num_active  n_windows
 89728         0.000299        15.992048     1509.0
 86369         0.000065              NaN        NaN
 40272         0.000181         3.655401     1509.0
 90523         0.000102         0.401590     1509.0
 86341         0.000923        98.966865     1509.0
 ...                ...              ...        ...
 54578         0.001389              NaN        NaN
 86416         0.000057              NaN        NaN
 89731         0.000538              NaN        NaN
 76083         0.000203         4.962227     1509.0
 11018         0.001560              NaN        NaN
 
 [1620 rows x 3 columns],
 mean     NaN
 std      NaN
 median   NaN
 min      NaN
 max      NaN
 p05      NaN
 p95      NaN
 dtype: float64)

In [13]:
def summarize(s):
    #drop nas
    s = s.dropna()
    return pd.Series({
        "mean": s.mean(), "std": s.std(), "median": s.median(),
        "min": s.min(), "max": s.max(), "p05": s.quantile(0.05), "p95": s.quantile(0.95)
    })

# R2s
r2_in = summarize(summaries["r2_insample_stage1"])
r2_oos = summarize(summaries["r2_oos_stage1"])

# # Betas (absolute non-zero)
# for key, details in details_per_stock.items():




#     beta_cols = [c for c in details.columns if c.startswith("Lasso_")]
#     beta_vals = details[beta_cols].to_numpy().ravel()
#     beta_stats = summarize(pd.Series(np.abs(beta_vals[beta_vals != 0])))

# # Selection rate
# sel_rate = details.groupby("target_column")["num_nonzero_recalc"].apply(lambda x: (x > 0).mean())
# sel_rate_stats = summarize(sel_rate)

# Table
out = pd.DataFrame([r2_in, r2_oos], #, beta_stats, sel_rate_stats],
                   index=["In-sample $R^2$", "Out-of-sample $R^2$"]) #, "$\beta$ absolute value", "Selection rate"]

print(out.to_latex(float_format="%.4f"))



\begin{tabular}{lrrrrrrr}
\toprule
 & mean & std & median & min & max & p05 & p95 \\
\midrule
In-sample $R^2$ & 0.1418 & 0.1596 & 0.0863 & -0.0040 & 0.7417 & -0.0020 & 0.4816 \\
Out-of-sample $R^2$ & -0.0351 & 0.0444 & -0.0177 & -0.3481 & 0.0032 & -0.1270 & -0.0008 \\
\bottomrule
\end{tabular}



Second stage estimation results

In [14]:
#first we take into considerations only case in which stage 2 t-stat is larger than 1.96
positive_tstat_mask = summaries['kappa_tstat'] > 1.96
positive_summaries = summaries[positive_tstat_mask]

#print the proportion of stocks with positive stage 2 t-stat
print("Proportion of stocks with significant stage 2 t-stat: ", len(positive_summaries) / len(summaries))




Proportion of stocks with significant stage 2 t-stat:  0.1382716049382716


In [15]:
def summarize(s):
    return pd.Series({
        "mean": s.mean(), "std": s.std(), "median": s.median(),
        "min": s.min(), "max": s.max(), "p05": s.quantile(0.05), "p95": s.quantile(0.95)
    })

# Stage 2 stats
r2_in_2 = summarize(pd.to_numeric(positive_summaries["r2_insample_stage2"], errors="coerce"))
r2_oos_2 = summarize(pd.to_numeric(positive_summaries["r2_oos_stage2"], errors="coerce"))
kappa = summarize(pd.to_numeric(positive_summaries["kappa"], errors="coerce"))
kappa_t = summarize(pd.to_numeric(positive_summaries["kappa_tstat"], errors="coerce"))

# Table
out_stage2 = pd.DataFrame([r2_in_2, r2_oos_2, kappa, kappa_t],
                          index=["In-sample $R^2$ ", "Out-of-sample $R^2$", "$\hat{\kappa}$", "$t$-statistic of $\hat{\kappa}$"])

print(out_stage2.to_latex(float_format="%.4f"))

\begin{tabular}{lrrrrrrr}
\toprule
 & mean & std & median & min & max & p05 & p95 \\
\midrule
In-sample $R^2$  & 0.0031 & 0.0026 & 0.0025 & 0.0000 & 0.0180 & 0.0003 & 0.0082 \\
Out-of-sample $R^2$ & -0.0093 & 0.0349 & -0.0028 & -0.3171 & 0.0154 & -0.0277 & 0.0048 \\
$\hat{\kappa}$ & 0.4363 & 0.2777 & 0.3465 & 0.0933 & 0.9969 & 0.1270 & 0.9388 \\
$t$-statistic of $\hat{\kappa}$ & 13.2576 & 94.7289 & 3.0373 & 1.9713 & 1411.5605 & 2.0221 & 26.5070 \\
\bottomrule
\end{tabular}



<>:15: SyntaxWarning: invalid escape sequence '\h'
<>:15: SyntaxWarning: invalid escape sequence '\h'
<>:15: SyntaxWarning: invalid escape sequence '\h'
<>:15: SyntaxWarning: invalid escape sequence '\h'
/var/folders/hb/bc0srnt52zx6lbw4t2k_t_k80000gn/T/ipykernel_56529/1479291098.py:15: SyntaxWarning: invalid escape sequence '\h'
  index=["In-sample $R^2$ ", "Out-of-sample $R^2$", "$\hat{\kappa}$", "$t$-statistic of $\hat{\kappa}$"])
/var/folders/hb/bc0srnt52zx6lbw4t2k_t_k80000gn/T/ipykernel_56529/1479291098.py:15: SyntaxWarning: invalid escape sequence '\h'
  index=["In-sample $R^2$ ", "Out-of-sample $R^2$", "$\hat{\kappa}$", "$t$-statistic of $\hat{\kappa}$"])


# Comparison between topics popular in academic research and those picked by stage 1

In [20]:
df_academia = pd.read_csv("final_topics_from_papers.csv", index_col=0)

In [23]:
top_topics = df_academia['topic'].value_counts()

In [24]:
top_topics

topic
Earnings forecasts       317
Bear/bull market         237
Share payouts            211
Bond yields              148
Acquired investment      126
                        ... 
Trade agreements           1
Private/public sector      1
European sovereign         1
Soft drinks                1
Disease                    1
Name: count, Length: 81, dtype: int64